# Session 16 lab: Judging a model

You will compute accuracy, precision, recall and F1 **by hand with NumPy**, check them against scikit-learn, draw the ROC curve, try `class_weight`, find a column that **leaks** the answer, tune before you test, and use **stratified cross-validation**. Same 8,000 loans and the same split as Session 15. All data is fictional.

Rhythm: **PREDICT**, run, **READ**, **CHANGE**. Cells with `# your code here` hold a placeholder: replace that line with your answer and run the cells in order. The `check(...)` lines print NOT YET until your number is right. What you keep: this notebook and `model_comparison.csv`.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, re, zipfile, urllib.request
DATA_LINK = "https://drive.google.com/file/d/11Z5tr2iKuPROn5cyi9P7me6nd3naKfNp/view?usp=sharing"   # the Google Drive link to data.zip that your tutor shares (leave empty to upload data.zip yourself)
if not os.path.exists("orders.csv"):
    if DATA_LINK:
        m = re.search(r"/d/([A-Za-z0-9_-]+)|[?&]id=([A-Za-z0-9_-]+)", DATA_LINK)
        file_id = (m.group(1) or m.group(2)) if m else DATA_LINK
        urllib.request.urlretrieve("https://drive.google.com/uc?export=download&id=" + file_id, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data files ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks():
    n = len(_results); k = sum(_results.values())
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

### Preparation (the same as Sessions 14 and 15)

Run this cell. It prepares the 8,000 loans, one-hot encodes the text, splits 75 and 25 with `stratify=y`, and fits the logistic regression pipeline from last session. `y_prob` holds the predicted probability of default for each test loan. The helper `safe` lets a check say NOT YET instead of stopping the notebook.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

def safe(f):
    try:
        return f()
    except Exception:
        return np.nan

loans = pd.read_csv("loans.csv")
bad_age = (loans.age < 18) | (loans.age > 100)                      # two impossible ages
loans.loc[bad_age, "age"] = loans.loc[~bad_age, "age"].median()
loans["income_missing"] = loans.monthly_income_ugx.isna().astype(int)
loans["monthly_income_ugx"] = loans.monthly_income_ugx.fillna(loans.monthly_income_ugx.median())
# 1.15 is an illustrative assumption for interest and fees, not in the data
loans["instalment_to_income"] = loans.loan_amount_ugx * 1.15 / loans.term_months / loans.monthly_income_ugx
FEATURES = ["age", "monthly_income_ugx", "income_missing", "loan_amount_ugx", "term_months", "previous_loans",
            "previous_late_payments", "group_loan", "savings_balance_ugx", "has_mobile_money", "distance_to_branch_km", "instalment_to_income"]
X = pd.concat([loans[FEATURES], pd.get_dummies(loans[["gender", "occupation", "loan_purpose"]], dtype=int)], axis=1)
y = loans["default"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))   # C = 1.0 and an L2 penalty are the defaults
model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]
actual = (y_test == 1).to_numpy()
print(len(X_train), len(X_test), int(actual.sum()), "defaults in the test loans")

---
## Step 1: The baseline model and the accuracy paradox

Before judging a model, score the simplest one.

**PREDICT first.** A baseline model says "no default" for every loan. Will its accuracy on the test loans be above or below 0.80? How many defaults will it find?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.dummy import DummyClassifier

baseline_model = DummyClassifier(strategy="constant", constant=1)   # your code here: DummyClassifier(strategy="most_frequent")
baseline_model.fit(X_train, y_train)
baseline_acc = np.nan   # your code here: the accuracy of the baseline on X_test and y_test (hint: baseline_model.score)
baseline_flag = baseline_model.predict(X_test)
baseline_recall = float(((baseline_flag == 1) & actual).sum() / actual.sum())
print("baseline accuracy:", round(baseline_acc, 4), " recall:", baseline_recall)

In [ ]:
check("baseline accuracy", safe(lambda: baseline_acc), 0.8745, 0.001)
check("baseline recall", baseline_recall, 0, 0.001)

**READ the output.** The baseline has a high accuracy and a recall of zero. Why can accuracy not be the only number we report here?

> My answer: 

---
## Step 2: The confusion matrix, precision, recall and F1 by hand

Count the four cells with NumPy, apply the formulas, then ask scikit-learn for the same numbers.

**PREDICT first.** At a decision threshold of 0.20, will precision be higher or lower than recall? Write down a guess for both.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
flag = y_prob >= 0.20
tp = np.nan   # your code here: flagged AND really defaulted (flag & actual)
fp = np.nan   # your code here: flagged but repaid (flag & ~actual)
fn = np.nan   # your code here: not flagged but defaulted (~flag & actual)
tn = np.nan   # your code here: not flagged and repaid (~flag & ~actual)
accuracy = np.nan   # your code here: (tp + tn) / all four counts
precision = np.nan   # your code here: tp / (tp + fp)
recall = np.nan   # your code here: tp / (tp + fn)
f1 = np.nan   # your code here: 2 * precision * recall / (precision + recall)
print("TP", tp, "FP", fp, "FN", fn, "TN", tn)
print("accuracy", round(accuracy, 3), "precision", round(precision, 3), "recall", round(recall, 3), "F1", round(f1, 3))

Now ask scikit-learn. Notice that its confusion matrix puts the **negative class first**: `[[TN, FP], [FN, TP]]`.

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

print(confusion_matrix(y_test, flag))
print("precision", round(precision_score(y_test, flag), 3), "recall", round(recall_score(y_test, flag), 3), "F1", round(f1_score(y_test, flag), 3))

In [ ]:
check("true positives at 0.20", safe(lambda: tp), 108, 4)
check("false positives at 0.20", safe(lambda: fp), 170, 8)
check("false negatives at 0.20", safe(lambda: fn), 143, 4)
check("true negatives at 0.20", safe(lambda: tn), 1579, 8)
check("precision at 0.20", safe(lambda: precision), 0.38848920863309355, 0.02)
check("recall at 0.20", safe(lambda: recall), 0.4302788844621514, 0.02)
check("F1 at 0.20", safe(lambda: f1), 0.40831758034026466, 0.02)
check("your precision equals scikit-learn", safe(lambda: precision), precision_score(y_test, flag), 1e-9)
check("your recall equals scikit-learn", safe(lambda: recall), recall_score(y_test, flag), 1e-9)
check("your F1 equals scikit-learn", safe(lambda: f1), f1_score(y_test, flag), 1e-9)

**READ the output.** Compare your accuracy at 0.20 with the baseline accuracy from Step 1. Which is higher? Which model is more useful to a lender, and why?

> My answer: 

**CHANGE one thing.** Raise the decision threshold to 0.50. Store the new precision in `precision_05` and recall in `recall_05`. Predict first: which goes up and which goes down?

> What I saw: 

In [ ]:
flag_05 = y_prob >= 0.50
precision_05 = np.nan   # your code here: precision_score(y_test, flag_05)
recall_05 = np.nan   # your code here: recall_score(y_test, flag_05)
print("precision", round(precision_05, 3), "recall", round(recall_05, 3))
check("precision at 0.50", safe(lambda: precision_05), 0.7611940298507462, 0.04)
check("recall at 0.50", safe(lambda: recall_05), 0.20318725099601595, 0.02)

Try five thresholds in a table. The loop is given: read the table, then choose the threshold with the highest F1. (F1 at 0.20 and at 0.15 are close, so the check accepts either.)

In [ ]:
rows = []
for t in [0.5, 0.3, 0.2, 0.15, 0.1]:
    f = y_prob >= t
    rows.append({"threshold": t, "flagged": int(f.sum()), "precision": precision_score(y_test, f), "recall": recall_score(y_test, f), "f1": f1_score(y_test, f), "accuracy": accuracy_score(y_test, f)})
results = pd.DataFrame(rows).set_index("threshold")
print(results.round(3))
best_t = np.nan   # your code here: the threshold (index) with the highest f1, results["f1"].idxmax()
print("best F1 at threshold", best_t)

In [ ]:
check("threshold with the best F1 (0.15 or 0.20)", safe(lambda: best_t), 0.175, 0.03)

---
## Step 3: ROC curve, AUC and class_weight

AUC judges the ranking of the loans at every threshold at once. The ROC curve plots the true positive rate (recall, sensitivity) against the false positive rate.

**PREDICT first.** The AUC is the chance that a random defaulter gets a higher probability than a random repaid loan. Guess the AUC of this model: nearer 0.5, 0.75 or 0.95?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve
import matplotlib.pyplot as plt

auc = np.nan   # your code here: roc_auc_score(y_test, y_prob). Pass the probabilities, not the yes or no flags
# the same number by counting pairs: a defaulter against a repaid loan
pos, neg = y_prob[actual], y_prob[~actual]
auc_pairs = np.nan   # your code here: the share of (defaulter, repaid) pairs where the defaulter has the higher probability (ties count half)
print("AUC", round(auc, 3), " by pairs", round(auc_pairs, 3), " pairs:", len(pos) * len(neg))
print("Gini coefficient = 2 x AUC - 1 =", round(2 * auc - 1, 3))
fpr, tpr, _ = roc_curve(y_test, y_prob)
prec, rec, _ = precision_recall_curve(y_test, y_prob)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(fpr, tpr)
ax[0].plot([0, 1], [0, 1], linestyle="--")
ax[0].set_xlabel("false positive rate")
ax[0].set_ylabel("true positive rate (recall)")
ax[0].set_title("ROC curve")
ax[1].plot(rec, prec)
ax[1].axhline(actual.mean(), linestyle="--")
ax[1].set_xlabel("recall")
ax[1].set_ylabel("precision")
ax[1].set_title("Precision-recall curve")
plt.show()

In [ ]:
check("AUC", safe(lambda: auc), 0.7557944323335588, 0.01)
check("AUC by counting pairs equals roc_auc_score", safe(lambda: auc_pairs - auc) + 0.0, 0, 1e-6)

**READ the output.** Find the dashed lines. What does the diagonal in the ROC curve mean, and what does the dashed horizontal line in the precision-recall curve mean?

> My answer: 

**CHANGE one thing.** Fit the same pipeline with `class_weight="balanced"`. Predict first: will the AUC change a lot? Will more or fewer loans be flagged at a decision threshold of 0.50?

> What I saw: 

In [ ]:
weighted_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))   # your code here: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
weighted_model.fit(X_train, y_train)
y_prob_w = weighted_model.predict_proba(X_test)[:, 1]
flagged_w = np.nan   # your code here: the number of test loans with y_prob_w at or above 0.50
auc_w = roc_auc_score(y_test, y_prob_w)
recall_w = recall_score(y_test, y_prob_w >= 0.50)
print("flagged at 0.50 with balanced weights:", flagged_w, " plain model:", int((y_prob >= 0.50).sum()))
print("recall", round(recall_w, 3), " AUC", round(auc_w, 3))

In [ ]:
check("loans flagged with balanced weights", safe(lambda: flagged_w), 569, 30)
check("recall with balanced weights", recall_w, 0.6175298804780877, 0.04)
check("AUC with balanced weights", auc_w, 0.7562978503367889, 0.01)

---
## Step 4: The leak hunt

A model that looks too good to be true usually is. Find the column that gives the answer away.

**PREDICT first.** Session 15 left out the column `collection_calls`. If we add it, will the AUC go up a little or a lot? Write your guess.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
correlations = loans[FEATURES + ["collection_calls", "default"]].corr()["default"].drop("default").abs().sort_values(ascending=False)
print(correlations.head(4).round(3))
leak = ""   # your code here: the name of the variable with the strongest correlation with default
calls_by_outcome = loans.groupby("default")["collection_calls"].agg(["mean", "min", "max"])
print(calls_by_outcome.round(2))

In [ ]:
check("the leaking column is collection_calls", int(leak == "collection_calls"), 1, 0)
check("average calls on defaulted loans", calls_by_outcome.loc[1, "mean"], 4.552788844621514, 0.01)
check("fewest calls on a defaulted loan", calls_by_outcome.loc[1, "min"], 1, 0)

**READ the output.** Every defaulted loan has at least one call. When are collection calls made: before or after the loan goes bad? Why does that make this column unusable for a lender who must decide at approval?

> My answer: 

In [ ]:
X_leaky = X.copy()
X_leaky["collection_calls"] = loans["collection_calls"]
XL_train, XL_test, yL_train, yL_test = train_test_split(X_leaky, y, test_size=0.25, stratify=y, random_state=42)
leaky_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
leaky_model.fit(XL_train, yL_train)
y_prob_leaky = np.full(len(XL_test), 0.5)   # your code here: the probability of default from leaky_model.predict_proba(XL_test), column 1
auc_leaky = np.nan   # your code here: roc_auc_score of yL_test against y_prob_leaky
print("AUC honest", round(auc, 3), " AUC with the leak", round(auc_leaky, 3))

In [ ]:
check("AUC of the leaky model", safe(lambda: auc_leaky), 0.9957790336652248, 0.01)

**CHANGE one thing.** The leaky model would fail on a new applicant. Why? Write one sentence in your own words in the cell below.

> What I saw: 

> My sentence: 

---
## Step 5: Tune before you test, stratified cross-validation, and the table you keep

One split can be lucky. Five folds, each keeping the same default rate, give a mean and a spread. Settings are chosen on the training loans (validation set or cross-validation). The test loans are scored once.

**PREDICT first.** Will the five fold AUC scores for the honest model be nearly identical or differ by a few hundredths?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5)   # no shuffle: the loans are in date order, so each fold is a block of months
print("defaults in each fold of 1,600 loans:", [int(y.iloc[test].sum()) for _, test in cv.split(X, y)])
cv_honest = np.full(5, np.nan)   # your code here: cross_val_score(model, X, y, cv=cv, scoring="roc_auc")
cv_leaky = np.full(5, np.nan)   # your code here: the same with X_leaky
print("honest:", cv_honest.round(3), "mean", round(cv_honest.mean(), 3), "std", round(cv_honest.std(ddof=1), 3))
print("leaky: ", cv_leaky.round(3), "mean", round(cv_leaky.mean(), 3))

In [ ]:
check("defaults in each fold", [int(y.iloc[test].sum()) for _, test in cv.split(X, y)], [200, 201, 201, 201, 201], 0)
check("mean AUC, honest features", safe(lambda: cv_honest.mean()), 0.7571638364925094, 0.01)
check("mean AUC, with the leak", safe(lambda: cv_leaky.mean()), 0.9961513401541258, 0.01)

**READ the output.** Does cross-validation find the leak? Look at the leaky scores: are they unstable?

> My answer: 

**CHANGE one thing.** Tune before you test: run the same cross-validation on the TRAINING loans only (X_train, y_train). These 6,000 loans give folds of 1,200. Compare the mean with the test AUC from Step 3.

> What I saw: 

In [ ]:
cv_train = np.full(5, np.nan)   # your code here: cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc")
print("training loans only:", cv_train.round(3), "mean", round(cv_train.mean(), 3), "std", round(cv_train.std(ddof=1), 3), " test AUC:", round(auc, 3))
check("mean AUC on the training loans", safe(lambda: cv_train.mean()), 0.7477532791703331, 0.01)

Last, the **model comparison table** you keep: one row per model, with the evaluation metrics that matter for a lender. The function is given.

In [ ]:
def comparison_row(name, prob, threshold):
    f = prob >= threshold
    return {"model": name, "threshold": threshold, "precision": precision_score(y_test, f, zero_division=0), "recall": recall_score(y_test, f, zero_division=0),
            "f1": f1_score(y_test, f, zero_division=0), "auc": roc_auc_score(y_test, prob)}

rows = [comparison_row("Baseline: flag nobody", np.zeros(len(y_test)), 0.5),
        comparison_row("Logistic regression", y_prob, 0.20),
        comparison_row("Logistic regression, balanced weights", y_prob_w, 0.50)]
rows.append(comparison_row("your row here", np.zeros(len(y_test)), 0.5))   # your code here: add a fourth row for the leaky model: comparison_row("With collection_calls (leaked, do not use)", y_prob_leaky, 0.20)
comparison = pd.DataFrame(rows)
comparison.to_csv("model_comparison.csv", index=False)
print(comparison.round(3).to_string(index=False))

In [ ]:
check("rows in the comparison table", len(comparison), 4, 0)
check("AUC of the honest model in the table", safe(lambda: comparison.loc[1, "auc"]), 0.7557944323335588, 0.01)
check("AUC of the leaky model in the table", safe(lambda: comparison.loc[3, "auc"]), 0.9957790336652248, 0.01)

In [ ]:
show_checks()

### My verdict

Write three sentences: which model you would report and with which threshold, what the leaked column was and why it must go, and what the cross-validated AUC says.

**Download** `model_comparison.csv` from the Colab file panel and keep it with this notebook. Next session: forests, boosting and overfitting.